# Load data

In [1]:
from datasets import load_dataset

dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_review_All_Beauty", trust_remote_code=True)
print(dataset["full"][0])

{'rating': 5.0, 'title': 'Such a lovely scent but not overpowering.', 'text': "This spray is really nice. It smells really good, goes on really fine, and does the trick. I will say it feels like you need a lot of it though to get the texture I want. I have a lot of hair, medium thickness. I am comparing to other brands with yucky chemicals so I'm gonna stick with this. Try it!", 'images': [], 'asin': 'B00YQ6X8EO', 'parent_asin': 'B00YQ6X8EO', 'user_id': 'AGKHLEW2SOWHNMFQIJGBECAF7INQ', 'timestamp': 1588687728923, 'helpful_vote': 0, 'verified_purchase': True}


In [2]:
amazon_review = []

for i in range(len(dataset["full"])):
    review = dataset["full"][i]['text']
    amazon_review.append(review)

In [3]:
len(amazon_review)

701528

# Preprocess data

In [4]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [5]:
import re

# preprocess each text string 
preprocessed_texts = [preprocess_text(str(review)) for review in amazon_review]

# Embeddings

In [6]:
from sentence_transformers import SentenceTransformer

sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = sentence_model.encode(preprocessed_texts, show_progress_bar=False)

# BERTopic

load the libraries

In [7]:
%load_ext cuml.accel

from bertopic import BERTopic
from hdbscan import HDBSCAN
from umap import UMAP


[2025-05-02 11:28:32.816] [CUML] [info] cuML: Installed accelerator for sklearn.
[2025-05-02 11:28:34.618] [CUML] [info] cuML: Installed accelerator for umap.
[2025-05-02 11:28:34.622] [CUML] [info] cuML: Installed accelerator for hdbscan.
[2025-05-02 11:28:34.622] [CUML] [info] cuML: Successfully initialized accelerator.


build the initial topic modeling

In [8]:
# Create instances of GPU-accelerated UMAP and HDBSCAN
umap_model = UMAP(n_components=5, n_neighbors=15, min_dist=0.0)
hdbscan_model = HDBSCAN(min_samples=10, gen_min_span_tree=True, prediction_data=True)

# Pass the above models to be used in BERTopic
topic_model = BERTopic(umap_model=umap_model, hdbscan_model=hdbscan_model)
topics, probs = topic_model.fit_transform(preprocessed_texts, embeddings)

[2025-05-02 11:28:35.995] [CUML] [info] Building knn graph using nn descent
[2025-05-02 11:28:43.364] [CUML] [info] Transform can only be run with brute force. Using brute force.


get the topic keywords

In [29]:
# Get the topic keywords
topic_info = topic_model.get_topic_info()  # Returns a DataFrame with topic details
topic_info  # Includes Topic ID and top keywords

,Topic,Count,Name,Representation,Representative_Docs
0,-1,470796,-1_hair_my_skin_it,"[hair, my, skin, it, and, for, this, in, is, the]",[i have been using the moisturizer for a few w...
1,0,23200,0_lashes_mascara_eyeliner_eyelashes,"[lashes, mascara, eyeliner, eyelashes, lash, e...",[this is the best mascara i have ever used i h...
2,1,9547,1_el_muy_es_de,"[el, muy, es, de, que, la, lo, para, producto,...",[excelente yo usaba el perfume miss dior bloom...
3,2,5874,2_nails_polish_nail_coat,"[nails, polish, nail, coat, gel, coats, polish...","[its a great product for nails, this is not na..."
4,3,4932,3_colors_color_pigmented_pink,"[colors, color, pigmented, pink, shade, brown,...","[love all the colors, great colors love it, lo..."
...,...,...,...,...,...
1956,1955,5,1955_detangle_detangles_ideawas_workthank,"[detangle, detangles, ideawas, workthank, yank...",[awesome detangles hair that you keep in a mes...
1957,1956,5,1956_skindinavia_meltingsliding_jessi_decays,"[skindinavia, meltingsliding, jessi, decays, s...",[i have oily skin and it doesnt help the skin ...
1958,1957,5,1957_pouf_abrasive_sponge_showermuch,"[pouf, abrasive, sponge, showermuch, agate, sp...",[i have enjoyed the sponges every time i showe...
1959,1958,5,1958_timesdoes_messier_mitts_smashed,"[timesdoes, messier, mitts, smashed, attempt, ...",[i tried to use this piece of junk twice and n...


# New BERTopic

filter documents by dominant topics

In [31]:
filtered_docs = [doc for doc, topic in zip(preprocessed_texts, topics) if topic != -1]
filtered_embeds = [emb for emb, topic in zip(embeddings, topics) if topic != -1]
filtered_topics = [topic for topic in topics if topic != -1]

re-fit a new topic model with only non-noise docs

In [36]:
# Create instances of GPU-accelerated UMAP and HDBSCAN
umap_model = UMAP(n_components=5, n_neighbors=15, min_dist=0.0)
hdbscan_model = HDBSCAN(min_samples=10, gen_min_span_tree=True, prediction_data=True)

# Pass the above models to be used in BERTopic
new_topic_model = BERTopic(umap_model=umap_model, hdbscan_model=hdbscan_model)
new_topics, new_probs = new_topic_model.fit_transform(filtered_docs, np.array(filtered_embeds))

[2025-05-02 12:20:55.782] [CUML] [info] Building knn graph using nn descent
[2025-05-02 12:20:57.918] [CUML] [info] Transform can only be run with brute force. Using brute force.


new intertopic distance map

In [ ]:
# Visualize the topics
new_topic_model.visualize_topics()

Interactive DataMapPlot figure

In [ ]:
import datamapplot
import numpy as np

# Reduce your embeddings to 2-dimensions
reduced_embeddings = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine').fit_transform(np.array(filtered_embeds))

# Create an interactive DataMapPlot figure
new_topic_model.visualize_document_datamap(
    filtered_docs,
    topics = filtered_topics, 
    reduced_embeddings=reduced_embeddings, 
    interactive=True
)
